# 🔄 Módulo 6 (visual) — Conversão de modelos (HF → MLX e GGUF → MLX)

Acompanha o **Módulo 6** do guia. Aqui você vai:
1. Converter um modelo Hugging Face para **MLX com quantização 4-bit** e ver a
   **economia de espaço em disco** num gráfico;
2. Ler um arquivo **GGUF** com o MLX nativo e inspecionar os tensores;
3. Entender os limites (quais quantizações o MLX lê) e o caminho inverso
   (MLX → GGUF via llama.cpp).

> Abra com `make lab` a partir da raiz, com o `.venv` ativo. Usa o modelo
> minúsculo **SmolLM2-135M** para ser rápido (~270 MB de download na 1ª vez).

## 1. Setup

In [ ]:
import os
import shutil

import matplotlib.pyplot as plt
import mlx.core as mx

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(ROOT)

def dir_size_mb(path):
    total = 0
    for dp, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(dp, f))
    return total / 1024 / 1024

print("Projeto:", ROOT)

## 2. HF → MLX com quantização 4-bit

`mlx_lm.convert` baixa o modelo em `float16` e o reescreve em MLX. Com
`quantize=True`, os pesos vão para 4 bits — bem menor, com pequena perda de
qualidade. Vamos medir o tamanho **sem** e **com** quantização.

In [ ]:
from mlx_lm import convert

HF_REPO = "HuggingFaceTB/SmolLM2-135M-Instruct"
OUT_FP16 = "models/mlx/smollm2-135m-fp16"
OUT_4BIT = "models/mlx/smollm2-135m-4bit"

for d in (OUT_FP16, OUT_4BIT):
    shutil.rmtree(d, ignore_errors=True)

print("Convertendo sem quantização (fp16)...")
convert(hf_path=HF_REPO, mlx_path=OUT_FP16, quantize=False)

print("Convertendo com quantização 4-bit...")
convert(hf_path=HF_REPO, mlx_path=OUT_4BIT, quantize=True, q_bits=4, q_group_size=64)

size_fp16 = dir_size_mb(OUT_FP16)
size_4bit = dir_size_mb(OUT_4BIT)
print(f"\nfp16 : {size_fp16:6.1f} MB")
print(f"4-bit: {size_4bit:6.1f} MB  ({size_fp16/size_4bit:.1f}x menor)")

## 3. 📊 Economia de espaço com a quantização

In [ ]:
plt.figure(figsize=(6, 4))
bars = plt.bar(["fp16", "4-bit"], [size_fp16, size_4bit],
               color=["#888", "seagreen"])
plt.bar_label(bars, fmt="%.0f MB")
plt.title("Tamanho em disco — SmolLM2-135M")
plt.ylabel("MB")
plt.tight_layout()
plt.show()

## 4. Testar o modelo convertido (4-bit)

Convertido corretamente, ele já gera texto.

In [ ]:
from mlx_lm import generate, load

model, tok = load(OUT_4BIT)
msgs = [{"role": "user", "content": "What is machine learning? One sentence."}]
prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
_ = generate(model, tok, prompt=prompt, max_tokens=60, verbose=True)

## 5. GGUF → MLX: ler e inspecionar

O MLX lê `.gguf` nativamente com `mx.load(..., return_metadata=True)`.
Se não houver um GGUF em `models/gguf/`, baixamos um pequeno (SmolLM2 F16).

> ⚠️ **Limite importante:** o loader do MLX só dequantiza **F16, F32, Q8_0,
> Q4_0, Q4_1**. Arquivos em **k-quant** (Q4_K_M, Q6_K, etc.) falham com
> `gguf_tensor_to_f16 failed`. Nesse caso, baixe uma variante F16/Q8_0.

In [ ]:
import glob

ggufs = glob.glob("models/gguf/*.gguf")
if not ggufs:
    print("Nenhum GGUF encontrado. Baixando SmolLM2-135M F16 (~258 MB)...")
    from huggingface_hub import hf_hub_download
    hf_hub_download(repo_id="unsloth/SmolLM2-135M-Instruct-GGUF",
                    filename="SmolLM2-135M-Instruct-F16.gguf",
                    local_dir="models/gguf")
    ggufs = glob.glob("models/gguf/*.gguf")

gguf_path = ggufs[0]
print("Lendo:", gguf_path)
weights, metadata = mx.load(gguf_path, return_metadata=True)
print(f"Tensores: {len(weights)} | arquitetura: {metadata.get('general.architecture')}")
print("\nPrimeiros tensores:")
for k in list(weights.keys())[:6]:
    print(f"  {k:34s} {tuple(weights[k].shape)}  {weights[k].dtype}")

## 6. 📊 Distribuição dos tensores por camada

Um jeito visual de entender a estrutura do modelo: quantos parâmetros há em
cada tipo de tensor.

In [ ]:
import re
from collections import defaultdict

por_tipo = defaultdict(int)
for name, arr in weights.items():
    # agrupa por sufixo (ex.: attn_output.weight, ffn_down.weight)
    tipo = re.sub(r"^blk\.\d+\.", "", name)
    por_tipo[tipo] += arr.size

itens = sorted(por_tipo.items(), key=lambda kv: kv[1], reverse=True)[:10]
labels = [k for k, _ in itens]
vals = [v / 1e6 for _, v in itens]

plt.figure(figsize=(9, 4))
bars = plt.barh(labels[::-1], vals[::-1], color="slateblue")
plt.bar_label(bars, fmt="%.1fM")
plt.title("Parâmetros por tipo de tensor (top 10)")
plt.xlabel("milhões de parâmetros")
plt.tight_layout()
plt.show()

## 7. Caminho inverso: MLX/HF → GGUF (via llama.cpp)

O MLX **não exporta** GGUF. Para gerar um `.gguf` (ex.: para rodar no Ollama ou
no llama.cpp), use a ferramenta do llama.cpp. Estes são comandos de terminal,
**não** Python:

```bash
brew install llama.cpp

# a partir de uma pasta de modelo HF (ou de um LoRA já fundido):
python convert_hf_to_gguf.py models/finetuned/merged-model \
    --outfile models/gguf/meu-modelo.gguf --outtype q8_0
```

Resumo dos caminhos do sandbox:

| De → Para | Ferramenta |
|-----------|-----------|
| HF → MLX  | `mlx_lm.convert` (célula 2) |
| GGUF → MLX | `mx.load` (célula 5) — só F16/F32/Q8_0/Q4_0/Q4_1 |
| MLX/HF → GGUF | `llama.cpp` (`convert_hf_to_gguf.py`) |

## 8. 🧪 Exercícios

1. Converta um modelo diferente (ex.: `Qwen/Qwen2.5-0.5B-Instruct`) e compare a
   economia de espaço.
2. Teste `q_bits=8` vs `q_bits=4` na quantização — quanto muda o tamanho?
3. Baixe um GGUF **Q8_0** e um **Q4_K_M** do mesmo modelo e confirme que só o
   primeiro carrega com `mx.load`.

📝 Registre no `docs/03_DIARIO.md`.